# 1. Kiểm tra và trích xuất RLDD
CPU, Internet On. Attach RLDD Full + Face Landmarker. Đọc HUONG_DAN_KAGGLE_TUNG_BUOC.md. Không train trong notebook này.

In [ ]:
import sys, os, json, subprocess, urllib.request, zipfile
from pathlib import Path
INPUT, WORK = Path('/kaggle/input'), Path('/kaggle/working')
REV = '04aa1f5'
archive = WORK/'code20.zip'
urllib.request.urlretrieve(f'https://github.com/toafn11/driver-monitoring-system/archive/{REV}.zip', archive)
destination = WORK/f'code_{REV}'
with zipfile.ZipFile(archive) as z:
    for name in z.namelist():
        if not (destination/name).resolve().is_relative_to(destination.resolve()):
            raise ValueError('Unsafe archive path')
    z.extractall(destination)
CODE = next(destination.rglob('prepare20.py')).parents[1]
os.chdir(CODE)
sys.path.insert(0, str(CODE))
subprocess.run([sys.executable,'-m','pip','install','-q','-r',str(CODE/'requirements-kaggle20.txt')],check=True)
# BEGIN VERIFIED PROJECT ACTIVATION
import importlib, hashlib
EXPECTED_CODE_HASHES = {'training/preflight20.py': '20c0fde7e542fd6cd9aa7623d6254dd57b8b45d770c0d53a3384055e2eb8c8e6', 'training/prepare20.py': '3548b813e043d57c6e06850de96c23d57e56c7f26fec346c3444546b856a6343', 'training/video_timestamps20.py': '84d5f81cc1f2bd16312ba573af251fe4ec1899e37a26882e7c80bc97168de32e', 'training/recover_cache20.py': '94cf8c6e95031df2525f1baa2a7d9422ed6b9a5239a575f7ee94acbca416eade'}
for relative, expected in EXPECTED_CODE_HASHES.items():
    actual = hashlib.sha256((CODE/relative).read_bytes()).hexdigest()
    if actual != expected:
        raise RuntimeError(f'Code integrity mismatch: {relative}; re-import the notebook')
# Replacing sys.path alone does not replace modules retained by a notebook kernel.
for name in list(sys.modules):
    if name in ('training','core','models') or name.startswith(('training.','core.','models.')):
        del sys.modules[name]
sys.path.insert(0, str(CODE))
importlib.invalidate_caches()
for name in ('training.preflight20','training.prepare20','training.video_timestamps20'):
    module = importlib.import_module(name)
    actual_path = Path(module.__file__).resolve()
    expected_path = (CODE/Path(*name.split('.'))).with_suffix('.py').resolve()
    if actual_path != expected_path:
        raise RuntimeError(f'Wrong module loaded: {name}: {actual_path}')
    print('Verified module:', name, actual_path)
print('Verified timestamp version:', sys.modules['training.video_timestamps20'].TIMESTAMP_VERSION)
# END VERIFIED PROJECT ACTIVATION
print('Code revision VERIFIED:', REV)
import shutil
if not shutil.which('ffprobe'): raise RuntimeError('FFprobe is required; use a Kaggle image with FFmpeg installed')


In [ ]:
from training.prepare20 import rldd_manifest
folds = sorted(INPUT.rglob('Fold1_part1'))
if not folds: raise FileNotFoundError('Attach thuannn18022005/uta-rldd-full')
ROOT = folds[0].parent
models = sorted(INPUT.rglob('face_landmarker.task'))
if len(models) != 1: raise ValueError('Attach exactly one Face Landmarker model')
MODEL = models[0]
MANIFEST = WORK/'manifest20.csv'
rldd_manifest(ROOT, MANIFEST)
SAMPLE_FPS = 10  # Confirm against the audit; identical in all shards and build.
CACHE = WORK/'cache20_v4'
CACHE_SOURCES = sorted(INPUT.rglob('cache20_v4'))
print('Model:', MODEL, 'Prior cache inputs:', CACHE_SOURCES)


In [ ]:
# A fresh process prevents stale notebook modules from affecting the audit.
audit = subprocess.run([sys.executable, '-u', '-m', 'training.prepare20', 'preflight',
    '--manifest', str(MANIFEST), '--output', str(WORK/'inventory20.json')], cwd=CODE)
if audit.returncode not in (0, 2): raise RuntimeError('Audit process failed; inspect its log')
report = json.loads((WORK/'inventory20.json').read_text())
for video in report['videos']:
    if video['issues']: print('ERROR:', video['path'], video['issues'])
    if video.get('warnings'): print('WARNING:', video['path'], video['warnings'])
print('Suggested FPS:', report['suggested_fps'])
if report['failed']: raise ValueError('Resolve audit failures first; see inventory20.json')
if report['suggested_fps'] is None or SAMPLE_FPS > report['suggested_fps']:
    raise ValueError('Review inventory and choose a common FPS; update every notebook consistently')


## Trích xuất theo nhóm
Lượt kiểm tra đầu để RUN_EXTRACTION=False. Sau khi audit đạt, đặt True. Tạo 4 bản notebook riêng có SHARD_INDEX=0,1,2,3. Không đổi NUM_SHARDS hoặc FPS giữa các bản. Mỗi bản Save Version → Save & Run All để lưu cache.

In [ ]:
RUN_EXTRACTION = False
SHARD_INDEX, NUM_SHARDS = 0, 4
if RUN_EXTRACTION:
    subprocess.run([sys.executable, '-u', '-m', 'training.prepare20', 'cache',
        '--manifest', str(MANIFEST), '--model', str(MODEL), '--cache-dir', str(CACHE),
        '--sample-fps', str(SAMPLE_FPS), '--shard-index', str(SHARD_INDEX),
        '--num-shards', str(NUM_SHARDS), '--cache-sources', *map(str, CACHE_SOURCES)], cwd=CODE, check=True)
    (WORK/f'shard_{SHARD_INDEX}_complete.json').write_text(json.dumps(dict(
        sample_fps=SAMPLE_FPS, shard=SHARD_INDEX, num_shards=NUM_SHARDS, revision=REV)))
else:
    print('Audit only. Review inventory20.json before enabling extraction.')
